### goal of notebook: extract only nuclei that pass QC


In [ ]:
Sys.time()

In [ ]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(RColorBrewer))
suppressPackageStartupMessages(library(cowplot))
suppressPackageStartupMessages(library(gridExtra))
suppressPackageStartupMessages(library(ggsci))
suppressPackageStartupMessages(library(scales))
suppressPackageStartupMessages(library(viridis))
suppressPackageStartupMessages(library(tidyverse))

In [ ]:
addArchRGenome("hg38")
addArchRThreads(threads = 1)

In [ ]:
projdir <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples <- read.table(paste0(projdir, 'txt/samples.txt'), sep='\t')$V1

In [ ]:
proj_path <- paste0(projdir, 'atac/archr/raw/YS3_ambimux')
proj <- loadArchRProject(path = proj_path, force = FALSE, showLogo = TRUE)

In [ ]:
proj

### first, subset project to final QC

In [ ]:
meta_path <- paste0(projdir, '/csv/qc/atac/atac_qc_filtered.csv')
meta <- read.csv(meta_path, sep='\t')
dim(meta)

In [ ]:
head(meta)

In [ ]:
head(meta$archr_barcode)

In [ ]:
curr_barcodes <- rownames(as.data.frame(getCellColData(proj)))
head(curr_barcodes)

In [ ]:
###
b_mask = curr_barcodes %in% meta$archr_barcode
f_mask = meta$archr_barcode %in% curr_barcodes
sum(b_mask)
sum(f_mask)

In [ ]:
final_qc = meta[f_mask,]
head(final_qc$archr_barcode)

In [ ]:
rownames(final_qc) <- final_qc$archr_barcode
dim(final_qc)

In [ ]:
head(final_qc)

In [ ]:
head(final_qc$archr_barcode)

In [ ]:
out <- paste0(projdir, 'atac/archr/raw/YS3_filtered_qc')
proj <- subsetArchRProject(proj,
                          cells = final_qc$archr_barcode,
                          outputDirectory = out,
                          dropCells = FALSE,
                          force=TRUE)

In [ ]:
proj

In [ ]:
proj <- addCellColData(
  ArchRProj = proj,
  data = final_qc$assignment,
  name = 'assignment',
  cells = rownames(final_qc)
)

In [ ]:
proj

### apply dimensionality reduction now

In [ ]:
proj <- addIterativeLSI(
    ArchRProj = proj,
    useMatrix = "TileMatrix", 
    name = "IterativeLSI", 
    iterations = 2, 
    clusterParams = list( #See Seurat::FindClusters
        resolution = c(0.2), 
        sampleCells = 10000, 
        n.start = 10
    ), 
    varFeatures = 25000, 
    dimsToUse = 1:30
)

In [ ]:
proj <- addClusters(
    input = proj,
    reducedDims = "IterativeLSI",
    method = "Seurat",
    name = "Clusters",
    resolution = 0.8
)

In [ ]:
proj <- addUMAP(
    ArchRProj = proj, 
    reducedDims = "IterativeLSI", 
    name = "UMAP", 
    nNeighbors = 30, 
    minDist = 0.5, 
    metric = "cosine"
)

In [ ]:
plotEmbedding(ArchRProj = proj,
              colorBy = "cellColData",
              name = "Clusters", embedding = "UMAP")

### write archr object + other metadata out

In [ ]:
outdir = paste0(projdir, 'atac/archr/processed/YS3_filtered_qc')
outdir

In [ ]:
saveArchRProject(
  ArchRProj = proj,
  outputDirectory = outdir,
  overwrite = TRUE,
  load = TRUE,
  dropCells = FALSE,
  logFile = createLogFile("saveArchRProject"),
  threads = getArchRThreads()
)

In [ ]:
embedding <- getEmbedding(ArchRProj = proj,
             embedding = "UMAP",
             returnDF = TRUE)
dim(embedding)

In [ ]:
out_embed = paste0(projdir, '/csv/atac/archr/umap_filtered.csv')
out_embed
write.table(embedding, out_embed, sep='\t', quote = FALSE)

In [ ]:
lsi <- getReducedDims(
      ArchRProj = proj,
      reducedDims = "IterativeLSI",
      returnMatrix = TRUE,
      dimsToUse = NULL,
      scaleDims = NULL,
      corCutOff = 0.75
    )
dim(lsi)

In [ ]:
out_lsi = paste0(projdir, '/csv/atac/archr/lsi_filtered.csv')
out_lsi
write.table(lsi, out_lsi, sep='\t', quote = FALSE)

In [ ]:
meta <- as.data.frame(getCellColData(proj))
out_meta = paste0(projdir, '/csv/atac/archr/meta_filtered.csv')
out_meta
write.table(meta, out_meta, sep='\t', quote = FALSE)

### duplicate everything for intersect RNA/ATAC

In [ ]:
proj_path <- paste0(projdir, 'atac/archr/raw/YS3_ambimux')
proj <- loadArchRProject(path = proj_path, force = FALSE, showLogo = TRUE)

In [ ]:
proj

In [ ]:
meta_path <- paste0(projdir, '/csv/qc/joint/joint_qc_filtered.csv')
meta <- read.csv(meta_path, sep='\t')
meta$archr_barcode <- paste0(meta$sample, '#', meta$barcode)

curr_barcodes <- rownames(as.data.frame(getCellColData(proj)))
b_mask = curr_barcodes %in% meta$archr_barcode
f_mask = meta$archr_barcode %in% curr_barcodes
final_qc = meta[f_mask,]
dim(final_qc)

In [ ]:
rownames(final_qc) <- final_qc$archr_barcode
out <- paste0(projdir, 'atac/archr/raw/YS3_intersect_rna')
proj <- subsetArchRProject(proj,
                          cells = final_qc$archr_barcode,
                          outputDirectory = out,
                          dropCells = FALSE,
                          force=TRUE)

In [ ]:
proj <- addIterativeLSI(
    ArchRProj = proj,
    useMatrix = "TileMatrix", 
    name = "IterativeLSI", 
    iterations = 2, 
    clusterParams = list( #See Seurat::FindClusters
        resolution = c(0.2), 
        sampleCells = 10000, 
        n.start = 10
    ), 
    varFeatures = 25000, 
    dimsToUse = 1:30
)

In [ ]:
proj <- addClusters(
    input = proj,
    reducedDims = "IterativeLSI",
    method = "Seurat",
    name = "Clusters",
    resolution = 0.8
)

In [ ]:
proj <- addUMAP(
    ArchRProj = proj, 
    reducedDims = "IterativeLSI", 
    name = "UMAP", 
    nNeighbors = 30, 
    minDist = 0.5, 
    metric = "cosine"
)

In [ ]:
plotEmbedding(ArchRProj = proj,
              colorBy = "cellColData",
              name = "Clusters", embedding = "UMAP")

In [ ]:
embedding <- getEmbedding(ArchRProj = proj,
             embedding = "UMAP",
             returnDF = TRUE)
dim(embedding)
out_embed = paste0(projdir, '/csv/atac/archr/umap_intersect_rna.csv')
write.table(embedding, out_embed, sep='\t', quote = FALSE)
lsi <- getReducedDims(
      ArchRProj = proj,
      reducedDims = "IterativeLSI",
      returnMatrix = TRUE,
      dimsToUse = NULL,
      scaleDims = NULL,
      corCutOff = 0.75
    )
dim(lsi)
out_lsi = paste0(projdir, '/csv/atac/archr/lsi_intersect_rna.csv')
write.table(lsi, out_lsi, sep='\t', quote = FALSE)

meta <- as.data.frame(getCellColData(proj))
out_meta = paste0(projdir, '/csv/atac/archr/meta_intersect_rna.csv')
write.table(meta, out_meta, sep='\t', quote = FALSE)

In [ ]:
outdir = paste0(projdir, 'atac/archr/processed/YS3_intersect_rna')
saveArchRProject(
  ArchRProj = proj,
  outputDirectory = outdir,
  overwrite = TRUE,
  load = TRUE,
  dropCells = FALSE,
  logFile = createLogFile("saveArchRProject"),
  threads = getArchRThreads()
)

In [ ]:
sessionInfo()

In [ ]:
Sys.time()